# SHA3 + 健康熵源：整合驗收
這份 Notebook 會重新設定 FPGA。請先確認已登記、目前由你獨占使用這張板子。

八個發布檔放在同一個**新的資料夾**。由上往下逐格 Shift+Enter；出錯就停止，不要 Run All 或重複下載電路。

資料流程：八路 RO（環形振盪器）→ 啟動與健康檢查 → 1024 bytes 原始資料 → 處理器搬運 → FPGA SHA3-256 → 32 bytes 摘要。每個摘要都會與 Python 比對。

這是功能整合；不是認證，也尚未證明每個 256-bit 輸出具有 256 bits 真實熵。

In [1]:
import hashlib, json, platform
import pynq
from trng_pynq import TrngSession, validate_artifacts
from trng_collect import collect
print("PYNQ", pynq.__version__, "Python", platform.python_version())
assert pynq.__version__.startswith("2.5"), "This release was checked with PYNQ 2.5"
print("kernel OK")

PYNQ 2.5 Python 3.6.5
kernel OK


## 1. 離線檢查（不碰硬體）
核對配對檔案、實際接線與發布紀錄。Notebook 會自動存檔，所以不列入不可變檔案的雜湊。

In [2]:
print(json.dumps(validate_artifacts("trng_sha3.bit"), indent=2))

{
  "build_id": "trng-integrated-20261115T162306Z",
  "clock_mhz": 25,
  "entropy_credit_validated": false,
  "certified": false
}


## 2. 填寫實驗紀錄
board_id 填實際登記板號；21600 只是沿用上次紀錄。experiment_date 必須改成你實際操作當天，格式為 YYYY-MM-DD。板上時間可能不準，不要直接拿它當實驗日期。

In [7]:
from datetime import datetime
board_id = "21600"
experiment_date = "2026-11-16"
assert board_id.strip()
datetime.strptime(experiment_date, "%Y-%m-%d")
print("Board:", board_id, "Experiment:", experiment_date)

Board: 21600 Experiment: 2026-11-16


## 3. 載入唯一一份整合電路
這格會設定 FPGA。預設 RO 關閉。請勿另外執行舊版 SHA3/Entropy Session，那些程式會載入另一份電路並覆蓋本版本。

In [8]:
session = TrngSession("trng_sha3.bit")
print(session.source.snapshot())

Verified release; downloading ONE integrated overlay (RO OFF)
Health source and SHA3 ready at 25 MHz
{'status': 0, 'count': 0, 'ro_enable': 0, 'fault_mask': 0, 'rct_mask': 0, 'apt_mask': 0, 'observed': 0, 'startup_seen': 0}


## 4. 先測 SHA3（34 次）
包括空訊息、abc、136-byte 區塊邊界和長訊息，各跑兩輪。全部必須 PASS。這些已知訊息只是功能測試，不是亂數。

In [9]:
assert session.self_test(rounds=2) == 34

PASS SHA3 empty 0
PASS SHA3 abc 3
PASS SHA3 pattern_1 1
PASS SHA3 pattern_3 3
PASS SHA3 pattern_4 4
PASS SHA3 pattern_7 7
PASS SHA3 pattern_31 31
PASS SHA3 pattern_32 32
PASS SHA3 pattern_64 64
PASS SHA3 pattern_135 135
PASS SHA3 pattern_136 136
PASS SHA3 pattern_137 137
PASS SHA3 pattern_271 271
PASS SHA3 pattern_272 272
PASS SHA3 pattern_273 273
PASS SHA3 pattern_1000 1000
PASS SHA3 pattern_4096 4096
PASS SHA3 empty 0
PASS SHA3 abc 3
PASS SHA3 pattern_1 1
PASS SHA3 pattern_3 3
PASS SHA3 pattern_4 4
PASS SHA3 pattern_7 7
PASS SHA3 pattern_31 31
PASS SHA3 pattern_32 32
PASS SHA3 pattern_64 64
PASS SHA3 pattern_135 135
PASS SHA3 pattern_136 136
PASS SHA3 pattern_137 137
PASS SHA3 pattern_271 271
PASS SHA3 pattern_272 272
PASS SHA3 pattern_273 273
PASS SHA3 pattern_1000 1000
PASS SHA3 pattern_4096 4096
SUCCESS 34 known-message FPGA SHA3 comparisons


## 5. 16 批端到端驗收
每批先丟棄 1024 個啟動樣本，再保留 1024 個八路樣本；健康檢查通過才送到 SHA3。成功後 RO 自動關閉。每批保留原始資料、健康狀態與摘要，不會只留下漂亮的輸出。

In [10]:
smoke = collect(session, board_id, experiment_date, count=16, divider=512)
print(json.dumps(smoke, indent=2))
assert smoke["summary"]["status"] == "PASS_FUNCTIONAL"

PASS integrated batches 16/16; RO OFF after each batch
Saved evidence: trng_results/2026-11-16_20261115T064314Z_8587ef6d.zip
{
  "archive": "trng_results/2026-11-16_20261115T064314Z_8587ef6d.zip",
  "summary": {
    "board_id": "21600",
    "experiment_date": "2026-11-16",
    "board_utc_timestamp": "20261115T064314Z_8587ef6d",
    "board_clock_may_be_wrong": true,
    "build_id": "trng-integrated-20261115T162306Z",
    "requested_batches": 16,
    "divider": 512,
    "status": "PASS_FUNCTIONAL",
    "certified": false,
    "entropy_credit_validated": false,
    "note": "RO restarts per batch; startup 1024 discarded; payload 1024 retained. PS bridges raw bytes to FPGA SHA3.",
    "completed_batches": 16,
    "raw_bytes": 16384,
    "digest_bytes": 512,
    "duplicate_raw_batches": 0,
    "duplicate_digests": 0,
    "all_hardware_digests_match_python": true,
    "health_faults": 0,
    "final_health": {
      "status": 194,
      "count": 1024,
      "ro_enable": 0,
      "fault_mask": 

## 6. 重新收集 1000 批（實際按下執行才會開始）
先確認上一格成功。本格可能需要幾分鐘，每 16 批會顯示進度。

新電路布局可能影響 RO，所以重新保留 1,024,000 bytes 原始八路資料和 1000×1000 重啟矩陣，供下一步統計評估。這一格成功表示每批健康檢查與硬體 SHA3 比對都通過；不等於熵評估或認證通過。

In [11]:
assert smoke["summary"]["status"] == "PASS_FUNCTIONAL"
campaign = collect(session, board_id, experiment_date, count=1000, divider=512)
print(json.dumps(campaign, indent=2))
assert campaign["summary"]["status"] == "PASS_FUNCTIONAL"

PASS integrated batches 16/1000; RO OFF after each batch
PASS integrated batches 32/1000; RO OFF after each batch
PASS integrated batches 48/1000; RO OFF after each batch
PASS integrated batches 64/1000; RO OFF after each batch
PASS integrated batches 80/1000; RO OFF after each batch
PASS integrated batches 96/1000; RO OFF after each batch
PASS integrated batches 112/1000; RO OFF after each batch
PASS integrated batches 128/1000; RO OFF after each batch
PASS integrated batches 144/1000; RO OFF after each batch
PASS integrated batches 160/1000; RO OFF after each batch
PASS integrated batches 176/1000; RO OFF after each batch
PASS integrated batches 192/1000; RO OFF after each batch
PASS integrated batches 208/1000; RO OFF after each batch
PASS integrated batches 224/1000; RO OFF after each batch
PASS integrated batches 240/1000; RO OFF after each batch
PASS integrated batches 256/1000; RO OFF after each batch
PASS integrated batches 272/1000; RO OFF after each batch
PASS integrated batc

## 7. 安全結束（也可在前面出錯後、介面仍有回應時執行）
確認 RO 關閉，並停止兩個 DMA 通道，再釋放傳輸記憶體。如果這格出錯，保留完整錯誤與 Notebook，停止後續操作；不要反覆重新載入電路。

In [12]:
session.close()
print("Final health:", session.source.snapshot())
tx_status = session.engine._mmio.read(4)
rx_status = session.engine._mmio.read(0x34)
print("DMA halted:", bool(tx_status & 1), bool(rx_status & 1))
assert tx_status & rx_status & 1
assert session.source.mmio.read(32) == 0
print("SUCCESS INTEGRATED BOARD FUNCTIONAL ACCEPTANCE; entropy assessment pending")

Final health: {'status': 194, 'count': 1024, 'ro_enable': 0, 'fault_mask': 0, 'rct_mask': 0, 'apt_mask': 0, 'observed': 2048, 'startup_seen': 1024}
DMA halted: True True
SUCCESS INTEGRATED BOARD FUNCTIONAL ACCEPTANCE; entropy assessment pending


## 8. 下載證據
按 Ctrl+S 儲存執行過的 Notebook。回到左上角 Jupyter 檔案頁，進入 trng_results，下載剛才 16 批與 1000 批各自產生的 ZIP（一次勾一個檔案 → Download）。另外從本 Notebook 的 File → Download as → Notebook (.ipynb) 下載含輸出的 Notebook。

請提供兩個 ZIP 與執行過的 Notebook，才能進行整合後原始熵分析。即使失敗也保留 FAILED_STOPPED 的 ZIP 與錯誤畫面，不要修改或刪除失敗批次紀錄。